# Principios básicos del clustering — Ejemplos

**Módulo 3 — Modelado y evaluación de enfoques no supervisados · Curso Analítica de Datos**

Este notebook acompaña las diapositivas [`3.7_Principios_basicos_del_clustering.pdf`](3.7_Principios_basicos_del_clustering.pdf) y lleva a código lo que allí se explica.

## Contenido

1. **Supervisado vs. no supervisado**: por qué una etiqueta de clúster es apenas un identificador, no una categoría validada.
2. **Qué caracteriza a un clúster**: coherencia interna y separación — y por qué no hay un único método universal.
3. **Observaciones, variables y preparación**: por qué reescalar antes de agrupar es indispensable.
4. **Similitud, distancia y vecinos**: cómo cambiar la medida cambia quién es el "vecino más cercano".
5. **Distancia euclídea y Manhattan**, con el ejemplo numérico exacto de las diapositivas.
6. **Medidas para distintos tipos de datos**: coseno, Jaccard, Hamming y Mahalanobis.
7. **Distancias para variables categóricas**: Ochiai, Phi, Russell-Rao, Parejas simples, Jaccard, Dice y Rogers-Tanimoto.
8. **Clustering con datos reales**: reproducimos el ejemplo de los 8 planetas del sistema solar de las diapositivas.

> 💡 Este notebook **no requiere descargar nada ni tener cuenta de Kaggle**: todos los datos se generan o se escriben directamente en el propio notebook (incluyendo los datos reales de los planetas).

In [ ]:
# Librerías que usaremos en todo el notebook
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.spatial.distance import euclidean, cityblock, cosine, jaccard, hamming, mahalanobis
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', 20)
plt.rcParams['figure.figsize'] = (7, 5)
rng = np.random.default_rng(42)

print('Librerías cargadas correctamente ✅')

---
## 1. Supervisado vs. no supervisado

Como en la diapositiva 2: en clasificación (supervisado), el modelo aprende a asignar **categorías conocidas y validadas**. En clustering (no supervisado) no hay etiquetas: el algoritmo solo propone **identificadores de grupo** — números sin significado propio. La interpretación ("el grupo 0 son clientes frecuentes", por ejemplo) la aporta el analista después, no el algoritmo.

In [ ]:
# Generamos un dataset sintético con 3 grupos reales, pero fingimos no conocer esa información
X_demo, _ = make_blobs(n_samples=90, centers=3, cluster_std=0.8, random_state=1)

modelo_demo = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X_demo)

fig, ax = plt.subplots()
dispersión = ax.scatter(X_demo[:, 0], X_demo[:, 1], c=modelo_demo.labels_, cmap='tab10')
plt.colorbar(dispersión, label='Identificador de grupo', ticks=[0, 1, 2])
ax.set_title('Salida de KMeans: 0, 1 y 2 son solo etiquetas arbitrarias')
plt.tight_layout()
plt.show()

print('Los números 0, 1 y 2 NO representan ningún orden ni magnitud: son solo nombres de grupo.')
print('Si volvemos a correr el algoritmo con otra semilla, el mismo grupo podría llamarse "2" en vez de "0".')

---
## 2. Qué caracteriza a un clúster

Como en la diapositiva 3: un clúster muestra **coherencia interna** (sus miembros se parecen entre sí) y **separación** respecto a otros grupos. Pero la estructura "visible" depende por completo de qué variables elegimos y cómo las representamos.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Caso 1: estructura clara (3 grupos bien separados)
X_claro, _ = make_blobs(n_samples=100, centers=3, cluster_std=0.6, random_state=5)
axes[0].scatter(X_claro[:, 0], X_claro[:, 1], alpha=0.6)
axes[0].set_title('Estructura clara: coherencia + separación')

# Caso 2: sin estructura real (puntos uniformemente distribuidos al azar)
X_sin_estructura = rng.uniform(-5, 5, size=(100, 2))
axes[1].scatter(X_sin_estructura[:, 0], X_sin_estructura[:, 1], alpha=0.6, color='#c62828')
axes[1].set_title('Sin estructura real: ¿cuántos grupos "ves" aquí?')

plt.tight_layout()
plt.show()

print('En el panel izquierdo, casi cualquier persona identificaría 3 grupos.')
print('En el derecho no hay ninguna estructura real, pero un algoritmo de clustering igual')
print('devolverá *algún* agrupamiento si se lo pedimos — por eso no basta con "correr el algoritmo":')
print('hay que preguntarse si la estructura encontrada tiene sentido real o es un artefacto.')

---
## 3. Observaciones, variables y preparación

Reproducimos la tabla de la diapositiva 4: tres clientes (A, B, C) con su número de compras al mes y su gasto mensual. El `ID` es una **etiqueta**, no una variable de similitud — nunca debe entrar al cálculo de distancias.

In [ ]:
clientes = pd.DataFrame({
    'ID': ['A', 'B', 'C'],
    'compras_mes': [2, 3, 10],
    'gasto_mensual': [80000, 100000, 700000],
})
clientes

In [ ]:
X_sin_escalar = clientes[['compras_mes', 'gasto_mensual']].values

dist_AC = euclidean(X_sin_escalar[0], X_sin_escalar[2])
dist_AB = euclidean(X_sin_escalar[0], X_sin_escalar[1])
print(f'Distancia A-C sin escalar: {dist_AC:,.0f}')
print(f'Distancia A-B sin escalar: {dist_AB:,.0f}')
print('\nLa distancia A-C está dominada casi por completo por el gasto mensual (en pesos,')
print('cientos de miles) — el número de compras (unidades) apenas pesa en la comparación.')

In [ ]:
X_escalado = StandardScaler().fit_transform(X_sin_escalar)

dist_AC_esc = euclidean(X_escalado[0], X_escalado[2])
dist_AB_esc = euclidean(X_escalado[0], X_escalado[1])
print(f'Distancia A-C estandarizada: {dist_AC_esc:.3f}')
print(f'Distancia A-B estandarizada: {dist_AB_esc:.3f}')
print('\nTras estandarizar (media 0, desviación estándar 1), ambas variables pesan por igual')
print('en la distancia — "parecerse" ahora refleja ambas variables, no solo la de mayor escala.')

---
## 4. Similitud, distancia y vecinos

Como advierte la diapositiva 5: **cambiar la medida de distancia puede cambiar quién es el vecino más cercano**. Lo comprobamos con un punto de referencia y tres candidatos.

In [ ]:
referencia = np.array([0, 0])
candidatos = {'P1 (0,3)': np.array([0, 3]), 'P2 (2,2)': np.array([2, 2]), 'P3 (3,1)': np.array([3, 1])}

print(f'{"candidato":12s} {"euclídea":>10s} {"manhattan":>10s}')
for nombre, punto in candidatos.items():
    print(f'{nombre:12s} {euclidean(referencia, punto):10.2f} {cityblock(referencia, punto):10.0f}')

print('\nCon distancia euclídea, el vecino más cercano de (0,0) es P2 (2,2).')
print('Con distancia Manhattan, el vecino más cercano es P1 (0,3) — ¡cambió!')
print('Esto ocurre porque Manhattan penaliza por igual cualquier desplazamiento en cada eje,')
print('mientras que euclídea favorece desplazamientos "diagonales" más cortos en línea recta.')

---
## 5. Distancia euclídea y Manhattan

Reproducimos el ejemplo exacto de la diapositiva 6: $P=(1,1)$, $Q=(4,5)$.

In [ ]:
P = np.array([1, 1])
Q = np.array([4, 5])

d_euclidea = euclidean(P, Q)
d_manhattan = cityblock(P, Q)

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([P[0], Q[0]], [P[1], Q[1]], color='#1565c0', marker='o', label=f'Euclídea = {d_euclidea:.0f}')
ax.plot([P[0], Q[0], Q[0]], [P[1], P[1], Q[1]], color='#ef6c00', linestyle='--',
        marker='o', label=f'Manhattan = {d_manhattan:.0f}')
ax.annotate('P (1,1)', P, textcoords='offset points', xytext=(5, -10))
ax.annotate('Q (4,5)', Q, textcoords='offset points', xytext=(5, 5))
ax.legend()
ax.set_title('Misma pareja de puntos, dos distancias distintas')
plt.tight_layout()
plt.show()

print(f'd_euclídea(P,Q) = sqrt((4-1)² + (5-1)²) = sqrt(9+16) = {d_euclidea:.0f}')
print(f'd_manhattan(P,Q) = |4-1| + |5-1| = 3 + 4 = {d_manhattan:.0f}')

---
## 6. Medidas para distintos tipos de datos

Como en las diapositivas 7 y 8, cada tipo de dato pide una medida distinta.

### 6.1 Coseno — orientación de vectores (documentos)

Compara la **dirección** de dos vectores, sin importar su magnitud. Clásico para comparar documentos representados por frecuencia de palabras (TF).

In [ ]:
# Dos "documentos" representados por cuántas veces mencionan 'chocolate' y 'candy'
documento_1 = np.array([5, 1])   # habla mucho de chocolate, poco de candy
documento_2 = np.array([1, 5])   # al revés

similitud_coseno = 1 - cosine(documento_1, documento_2)
print(f'Similitud coseno: {similitud_coseno:.3f}  (1 = misma dirección, 0 = direcciones perpendiculares)')
print('Aunque ninguno de los dos vectores es igual al otro, ambos "hablan sobre dulces" en algún grado;')
print('el coseno capta eso mejor que una distancia euclídea, que sí se vería afectada por qué tan')
print('largo es cada documento (cuántas palabras en total).')

### 6.2 Jaccard — intersección sobre unión (conjuntos)

Compara **qué proporción de los elementos combinados tienen en común** dos conjuntos. Clásico para canastas de productos comprados.

In [ ]:
compras_cliente1 = {'leche', 'pan', 'huevos'}
compras_cliente2 = {'pan', 'huevos', 'queso', 'jamón'}

interseccion = compras_cliente1 & compras_cliente2
union = compras_cliente1 | compras_cliente2
similitud_jaccard = len(interseccion) / len(union)

print(f'Intersección: {interseccion}')
print(f'Unión: {union}')
print(f'Similitud de Jaccard: {len(interseccion)}/{len(union)} = {similitud_jaccard:.3f}')

### 6.3 Hamming — posiciones distintas (códigos binarios)

Cuenta en cuántas posiciones difieren dos secuencias de la misma longitud.

In [ ]:
codigo_1 = np.array([0, 1, 1, 0, 0, 1, 0])
codigo_2 = np.array([0, 0, 1, 0, 1, 1, 1])

posiciones_distintas = (codigo_1 != codigo_2).sum()
print(f'Códigos:     {codigo_1}  vs  {codigo_2}')
print(f'Posiciones distintas (distancia de Hamming): {posiciones_distintas} de {len(codigo_1)}')

### 6.4 Mahalanobis — distancia que considera la covarianza

Cuando las variables numéricas están correlacionadas (diapositiva 8), la distancia euclídea puede ser engañosa: dos puntos a la misma distancia euclídea del centro pueden ser muy distintos en términos de qué tan "típicos" son, según la forma de la nube de datos.

In [ ]:
# Datos numéricos correlacionados (como en el diagrama de la diapositiva 8)
X_corr = rng.multivariate_normal([0, 0], [[4, 3], [3, 4]], 300)
centro = X_corr.mean(axis=0)
matriz_covarianza = np.cov(X_corr.T)
inv_covarianza = np.linalg.inv(matriz_covarianza)

# Un punto "atípico": no muy lejos del centro en línea recta, pero en una dirección
# que la nube de datos casi nunca toma (va en contra de la correlación positiva)
punto_atipico = np.array([3, -3])

d_euclidea = euclidean(centro, punto_atipico)
d_mahalanobis = mahalanobis(centro, punto_atipico, inv_covarianza)

fig, ax = plt.subplots()
ax.scatter(X_corr[:, 0], X_corr[:, 1], alpha=0.3, s=10)
ax.scatter(*centro, color='black', marker='x', s=80, label='Centro')
ax.scatter(*punto_atipico, color='red', marker='D', s=80, label='Punto a evaluar')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Distancia euclídea del punto al centro: {d_euclidea:.2f}')
print(f'Distancia de Mahalanobis (considera la covarianza): {d_mahalanobis:.2f}')
print('\nMahalanobis "sabe" que los datos tienden a crecer juntos (correlación positiva),')
print('así que un punto que va en contra de esa tendencia resulta más atípico de lo que')
print('sugeriría solo mirar su distancia euclídea en línea recta.')

---
## 7. Distancias para variables categóricas

Como en la diapositiva 9, para variables binarias (presente/ausente) se arma una tabla de contingencia con $a$ (presente en ambos), $b$ y $c$ (presente en solo uno) y $d$ (ausente en ambos), y se combinan de distintas formas.

In [ ]:
def tabla_contingencia(x, y):
    """a, b, c, d según la tabla de la diapositiva 9, para dos vectores binarios."""
    x, y = np.asarray(x, dtype=bool), np.asarray(y, dtype=bool)
    a = np.sum(x & y)          # presente en ambos
    b = np.sum(x & ~y)         # presente solo en x
    c = np.sum(~x & y)         # presente solo en y
    d = np.sum(~x & ~y)        # ausente en ambos
    return a, b, c, d

def medidas_categoricas(x, y):
    a, b, c, d = tabla_contingencia(x, y)
    m = a + b + c + d
    return {
        'Ochiai':           a / np.sqrt((a + b) * (a + c)),
        'Phi':              (a * d - b * c) / np.sqrt((a + b) * (c + d) * (a + c) * (b + d)),
        'Russell-Rao':      a / m,
        'Parejas simples':  (a + d) / m,
        'Jaccard':          a / (a + b + c),
        'Dice':             2 * a / (2 * a + b + c),
        'Rogers-Tanimoto':  (a + d) / (a + d + 2 * (b + c)),
    }

# Dos clientes, cada posición indica si compraron (1) o no (0) uno de 7 productos
cliente_1 = [1, 0, 1, 1, 0, 0, 1]
cliente_2 = [1, 1, 1, 0, 0, 0, 1]

for nombre, valor in medidas_categoricas(cliente_1, cliente_2).items():
    print(f'  {nombre:18s} = {valor:.3f}')

---
## 8. Clustering con datos reales: ¿qué planetas se parecen entre sí?

Cerramos reproduciendo el ejemplo de la diapositiva 10: agrupar los 8 planetas del sistema solar usando solo su **diámetro ecuatorial** y su **densidad media** (valores reales, del *NASA Planetary Fact Sheet*).

In [ ]:
planetas = pd.DataFrame({
    'planeta':      ['Mercurio', 'Venus', 'Tierra', 'Marte', 'Júpiter', 'Saturno', 'Urano', 'Neptuno'],
    'diametro_km':  [4879, 12104, 12756, 6792, 142984, 120536, 51118, 49528],
    'densidad':     [5.427, 5.243, 5.514, 3.933, 1.326, 0.687, 1.270, 1.638],
})
planetas

In [ ]:
# Igual que en la sección 3: estandarizamos antes de calcular distancias, porque el
# diámetro (miles de km) y la densidad (g/cm³) están en escalas completamente distintas.
X_planetas = planetas[['diametro_km', 'densidad']].values
X_planetas_esc = StandardScaler().fit_transform(X_planetas)

kmeans_planetas = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X_planetas_esc)
planetas['grupo'] = kmeans_planetas.labels_

for grupo in sorted(planetas['grupo'].unique()):
    miembros = planetas.loc[planetas['grupo'] == grupo, 'planeta'].tolist()
    print(f'Grupo {grupo}: {", ".join(miembros)}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
dispersión = ax.scatter(planetas['diametro_km'], planetas['densidad'],
                         c=planetas['grupo'], cmap='viridis', s=120)
for _, fila in planetas.iterrows():
    ax.annotate(fila['planeta'], (fila['diametro_km'], fila['densidad']),
                textcoords='offset points', xytext=(8, 5))
ax.set_xlabel('Diámetro ecuatorial (km)')
ax.set_ylabel('Densidad media (g/cm³)')
ax.set_title('K-means (K=3) sobre los 8 planetas')
plt.tight_layout()
plt.show()

ratio = planetas.loc[planetas['planeta']=='Tierra','densidad'].values[0] / \
        planetas.loc[planetas['planeta']=='Saturno','densidad'].values[0]
print(f'Dato curioso: la Tierra es unas {ratio:.1f} veces más densa que Saturno.')

Los tres grupos que aparecen son, igual que en la diapositiva: **pequeños y densos** (Mercurio, Venus, Tierra, Marte — los planetas rocosos), **diámetro intermedio** (Urano, Neptuno) y **los mayores diámetros** (Júpiter, Saturno). Observa que los *números* de grupo (0, 1, 2) que asignó `KMeans` no tienen por qué coincidir con el orden "Grupo 1/2/3" de las diapositivas — es justo el punto de la sección 1: **son identificadores arbitrarios**, la interpretación ("rocosos", "gigantes gaseosos"...) la ponemos nosotros.

---
## 9. Para pensar y discutir en clase

1. En la sección 8, ¿cambiarían los grupos de planetas si usáramos otras características (por ejemplo, distancia al sol y número de lunas, en vez de diámetro y densidad)? ¿Por qué?
2. En la sección 3 (clientes), ¿qué otra variable podrías agregar que, sin reescalar, también dominara la distancia por su magnitud?
3. Elige dos de las medidas de la sección 7 (Ochiai, Phi, Russell-Rao, Parejas simples, Jaccard, Dice, Rogers-Tanimoto) y explica, con tus palabras, en qué se diferencian — ¿en qué situación dos clientes podrían obtener un puntaje muy distinto en una medida y muy similar en otra?

No hay una única respuesta "correcta": lo importante es justificar cada respuesta con evidencia del propio notebook.

---
## Cierre

En este notebook llevamos a código las ideas de las diapositivas [`3.7_Principios_basicos_del_clustering.pdf`](3.7_Principios_basicos_del_clustering.pdf):

- Por qué una etiqueta de clúster es solo un **identificador**, no una categoría validada.
- Qué hace que un grupo de puntos se perciba como un **clúster real**: coherencia + separación.
- Por qué **preparar los datos** (reescalar, tratar los ID como etiquetas) define qué significa "parecerse".
- Cómo **cambiar la medida de distancia cambia la vecindad**, con el ejemplo exacto de euclídea vs. Manhattan.
- Medidas para distintos tipos de datos: **coseno**, **Jaccard**, **Hamming**, **Mahalanobis**, y siete medidas específicas para **variables categóricas**.
- Un ejemplo de clustering con **datos reales**: agrupar los planetas del sistema solar.

### Recursos adicionales
- [Documentación de `scipy.spatial.distance`](https://docs.scipy.org/doc/scipy/reference/spatial.distance.html)
- [Documentación de `sklearn.cluster.KMeans`](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html)
- [NASA Planetary Fact Sheet (fuente de los datos de los planetas)](https://nssdc.gsfc.nasa.gov/planetary/factsheet/)